# Direct BigQuery client query

Reads `data/input/seeds.csv` and performs a parameterized `IN UNNEST` query using `google.cloud.bigquery` directly.

In [ ]:
from pathlib import Path
import pandas as pd
from google.cloud import bigquery

BASE = Path.cwd().parent
INPUT_DIR = BASE / 'data' / 'input'
df = pd.read_csv(INPUT_DIR / 'seeds.csv')
txids = df['txid'].dropna().tolist()
len(txids)

In [ ]:
client = bigquery.Client()
sql = (
    "SELECT TO_HEX(`hash`) AS txid, block_timestamp\n"
    "FROM `bigquery-public-data.crypto_bitcoin.transactions`\n"
    "WHERE TO_HEX(`hash`) IN UNNEST(@txid_list)\n"
)
job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter('txid_list', 'STRING', txids)])
rows = client.query(''.join(sql), job_config=job_config).result()
result_map = {row['txid']: row['block_timestamp'] for row in rows}
df['block_timestamp'] = df['txid'].map(result_map)
df.head()